In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_validate
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, ConfusionMatrixDisplay,
    RocCurveDisplay, confusion_matrix
)
from preprocessing import load_data, split_data, make_preprocessor, FEATURES, RANDOM_STATE

FIG_DIR = Path("results/logistic_regression")
FIG_DIR.mkdir(parents=True, exist_ok=True)
print("Libraries imported successfully.")

Libraries imported successfully.


In [3]:
X, y = load_data()
X_train, X_test, y_train, y_test = split_data(X, y)
print("Total records:", len(X))
print("Training records:", len(X_train), "Test records:", len(X_test))
print("Training diabetes rate:", round(y_train.mean(), 3))
print("Test diabetes rate:", round(y_test.mean(), 3))
print("Missing predictor values (after zero replacement):")
display(X.isna().sum())

Total records: 768
Training records: 614 Test records: 154
Training diabetes rate: 0.349
Test diabetes rate: 0.351
Missing predictor values (after zero replacement):


Preg          0
Glucose       5
BP           35
Skin        227
Insulin     374
BMI          11
Pedigree      0
Age           0
dtype: int64

In [4]:
def make_lr_pipeline(C=1.0, class_weight=None):
    return Pipeline([
        ("preprocess", make_preprocessor(scale=True)),
        ("lr", LogisticRegression(
            C=C, class_weight=class_weight, max_iter=2000,
            random_state=RANDOM_STATE
        ))
    ])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc"
}
baseline_scores = cross_validate(
    make_lr_pipeline(), X_train, y_train,
    scoring=scoring, cv=cv, n_jobs=1
)
baseline_summary = pd.Series({
    metric: baseline_scores[f"test_{metric}"].mean() for metric in scoring
})
print("Baseline five-fold CV metrics (training data only):")
display(baseline_summary.round(3).to_frame("Mean CV score"))

Baseline five-fold CV metrics (training data only):


,Mean CV score
accuracy,0.765
precision,0.715
recall,0.546
f1,0.615
roc_auc,0.821


In [5]:
grid = GridSearchCV(
    make_lr_pipeline(),
    param_grid={
        "lr__C": [0.01, 0.1, 1, 10, 100],
        "lr__class_weight": [None, "balanced"]
    },
    scoring=scoring,
    refit="roc_auc",
    cv=cv,
    n_jobs=1,
    return_train_score=True
)
grid.fit(X_train, y_train)
print("Best hyperparameters:", grid.best_params_)
print("Best CV ROC-AUC:", round(grid.best_score_, 3))

best_idx = grid.best_index_
tuned_summary = pd.Series({
    metric: grid.cv_results_[f"mean_test_{metric}"][best_idx]
    for metric in scoring
})
comparison = pd.DataFrame({
    "Baseline CV": baseline_summary,
    "Tuned CV": tuned_summary
}).round(3)
display(comparison)

# The tuned CV scores were used for selection and can be optimistic;
# the independent test set below provides the final evaluation.

Best hyperparameters: {'lr__C': 0.1, 'lr__class_weight': 'balanced'}
Best CV ROC-AUC: 0.825


,Baseline CV,Tuned CV
accuracy,0.765,0.738
precision,0.715,0.615
recall,0.546,0.682
f1,0.615,0.645
roc_auc,0.821,0.825


In [6]:
best_model = grid.best_estimator_
y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)[:, 1]

test_metrics = pd.Series({
    "accuracy": accuracy_score(y_test, y_pred),
    "precision": precision_score(y_test, y_pred, zero_division=0),
    "recall": recall_score(y_test, y_pred, zero_division=0),
    "f1": f1_score(y_test, y_pred, zero_division=0),
    "roc_auc": roc_auc_score(y_test, y_prob)
}, name="Held-out test")
print("Final test metrics:")
display(test_metrics.round(3).to_frame())
print("Classification report:")
print(classification_report(y_test, y_pred, target_names=["No diabetes", "Diabetes"], zero_division=0))
print("Confusion matrix [actual rows, predicted columns]:")
print(confusion_matrix(y_test, y_pred, labels=[0, 1]))

Final test metrics:


,Held-out test
accuracy,0.766
precision,0.636
recall,0.778
f1,0.700
roc_auc,0.865


Classification report:
              precision    recall  f1-score   support

 No diabetes       0.86      0.76      0.81       100
    Diabetes       0.64      0.78      0.70        54

    accuracy                           0.77       154
   macro avg       0.75      0.77      0.75       154
weighted avg       0.78      0.77      0.77       154

Confusion matrix [actual rows, predicted columns]:
[[76 24]
 [12 42]]
